In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from alerce.core import Alerce

# ==========================================
# 1. Fetch and Clean Swift/BAT X-ray Data
# ==========================================
swift_file_path = "https://swift.gsfc.nasa.gov/results/transients/weak/SWIFTJ1727.8-1613.lc.txt"
swift_cols = ["TIME", "RATE", "ERROR", "YEAR", "DAY", "STAT_ERR", "SYS_ERR", "DATA_FLAG"]

swift_df = pd.read_csv(
    swift_file_path, sep=r"\s+", comment="#", names=swift_cols, usecols=range(8), header=None
)

for col in ["TIME", "RATE", "ERROR", "DATA_FLAG"]:
    swift_df[col] = pd.to_numeric(swift_df[col], errors="coerce")
    
swift_df = swift_df.dropna(subset=["TIME", "RATE", "ERROR"])

# CORRECTION: Filter out bad data flags and negative rates
swift_df = swift_df[(swift_df["DATA_FLAG"] == 0) & (swift_df["RATE"] > 0)].copy()

# CONVERSION: Swift BAT Rate to Flux (erg/cm^2/s)
conversion_factor = 2.4e-8 / 0.22
swift_df["X_FLUX"] = swift_df["RATE"] * conversion_factor
swift_df["X_FLUX_ERR"] = swift_df["ERROR"] * conversion_factor

# Isolate the outburst window for reliable fitting (roughly MJD 60150 to 60400)
outburst_mask = (swift_df["TIME"] > 60150) & (swift_df["TIME"] < 60400)
swift_outburst = swift_df[outburst_mask]

# ==========================================
# 2. Fetch and Clean ZTF Optical Data
# ==========================================
print("Fetching ZTF data from ALeRCE...")
alerce = Alerce()
ra, dec = 261.9305, -16.2052
objects = alerce.query_objects(ra=ra, dec=dec, radius=15, format="pandas")
oid = objects["oid"].iloc[0]
ztf_df = alerce.query_detections(oid, format="pandas")

mag_col = "magpsf" if "magpsf" in ztf_df.columns else "mag"
err_col = "sigmapsf" if "sigmapsf" in ztf_df.columns else "e_mag"

# Clean NaN values
ztf_df = ztf_df.dropna(subset=["mjd", mag_col, err_col])

# CONVERSION: AB Magnitude to Flux Density (mJy)
def mag_to_flux(mag, mag_err):
    flux = 3631 * 10**(-0.4 * mag) * 1000  # in mJy
    flux_err = flux * (mag_err / 1.0857)
    return flux, flux_err

ztf_df["OPT_FLUX"], ztf_df["OPT_FLUX_ERR"] = mag_to_flux(ztf_df[mag_col], ztf_df[err_col])

g_band = ztf_df[ztf_df["fid"] == 1]
r_band = ztf_df[ztf_df["fid"] == 2]

# ==========================================
# 3. Model Fitting & Simulation
# ==========================================
def fred_model(t, t_start, t_rise, t_decay, amplitude):
    """FRED model for X-ray lightcurve fitting"""
    flux = np.zeros_like(t)
    mask = t > t_start
    t_eff = t[mask] - t_start
    norm_factor = np.exp(2 * np.sqrt(t_rise / t_decay))
    flux[mask] = amplitude * norm_factor * np.exp(-(t_rise / t_eff) - (t_eff / t_decay))
    return flux

# Fit the FRED model to the X-ray data
# Guesses: [Start Time, Rise Time, Decay Time, Peak Amplitude]
initial_guesses = [60170, 5.0, 30.0, swift_outburst["X_FLUX"].max()]
bounds = ([60150, 0.1, 1.0, 0], [60200, 50.0, 200.0, 1e-6])

popt, _ = curve_fit(
    fred_model, 
    swift_outburst["TIME"], 
    swift_outburst["X_FLUX"], 
    p0=initial_guesses, 
    bounds=bounds
)

# Generate a smooth array of times for plotting the models
model_time = np.linspace(60100, 60600, 1000)
xray_fit_curve = fred_model(model_time, *popt)

# SIMULATE OPTICAL: F_opt scales with F_x^0.5 (Standard Reprocessing)
# We apply a scaling factor to roughly match the r-band mJy magnitude range
scaling_factor = 1.2e5 
simulated_opt_curve = scaling_factor * (xray_fit_curve ** 0.5)

# ==========================================
# 4. Plotting Multiwavelength Data & Models
# ==========================================
fig, ax1 = plt.subplots(figsize=(14, 8))

# X-ray Data and Fit
color1 = 'dimgray'
ax1.errorbar(
    swift_df["TIME"], swift_df["X_FLUX"], yerr=swift_df["X_FLUX_ERR"],
    fmt="o", color=color1, ms=3, alpha=0.5, label="Swift/BAT Data (15-50 keV)"
)
ax1.plot(
    model_time, xray_fit_curve, color="black", linewidth=2.5, 
    label=f"X-ray FRED Fit\n($t_{{rise}}$={popt[1]:.1f}d, $t_{{decay}}$={popt[2]:.1f}d)"
)

ax1.set_xlabel("Time (Modified Julian Date - MJD)", fontsize=12)
ax1.set_ylabel("X-ray Flux (erg cm$^{-2}$ s$^{-1}$)", color="black", fontsize=12)
ax1.tick_params(axis="y", labelcolor="black")
ax1.grid(True, linestyle="--", alpha=0.5)

# Optical Data and Simulation
ax2 = ax1.twinx()
color2 = 'darkred'

if not g_band.empty:
    ax2.errorbar(
        g_band["mjd"], g_band["OPT_FLUX"], yerr=g_band["OPT_FLUX_ERR"],
        fmt="o", color="forestgreen", ms=5, alpha=0.7, label="ZTF g-band Data"
    )
if not r_band.empty:
    ax2.errorbar(
        r_band["mjd"], r_band["OPT_FLUX"], yerr=r_band["OPT_FLUX_ERR"],
        fmt="o", color="firebrick", ms=5, alpha=0.7, label="ZTF r-band Data"
    )

ax2.plot(
    model_time, simulated_opt_curve, color="orange", linewidth=2.5, linestyle="--",
    label="Simulated Optical (Reprocessing $F_X^{0.5}$)"
)

ax2.set_ylabel("Optical Flux Density (mJy)", color=color2, fontsize=12)
ax2.tick_params(axis="y", labelcolor=color2)

# Focus the plot window on the outburst
ax1.set_xlim(60150, 60400)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=11, framealpha=0.9)

plt.title("SWIFT J1727.8-1613: X-ray FRED Fit & Optical Reprocessing Simulation", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from alerce.core import Alerce

# ==========================================
# 1. Fetch and Clean MAXI X-ray Data
# ==========================================
print("Fetching MAXI data...")
maxi_url = "https://maxi.riken.jp/star_data/J1727-162/J1727-162_g_lc_1day_all.dat"

# MAXI columns: MJD, Rate(2-20keV), Err, Rate(2-4), Err, Rate(4-10), Err, Rate(10-20), Err
maxi_cols = ["MJD", "RATE_2_20", "ERR_2_20", "R1", "E1", "R2", "E2", "R3", "E3"]
maxi_df = pd.read_csv(maxi_url, sep=r"\s+", names=maxi_cols, comment="#")

# Filter for valid data during the 2023 outburst
maxi_df = maxi_df[(maxi_df["MJD"] > 60150) & (maxi_df["MJD"] < 60400)].copy()
maxi_df = maxi_df.dropna(subset=["MJD", "RATE_2_20", "ERR_2_20"])
maxi_df = maxi_df[maxi_df["RATE_2_20"] > 0] # Remove negative background fluctuations

# CONVERSION: MAXI Rate to Flux (erg/cm^2/s)
# 1 Crab in MAXI (2-20 keV) ~ 3.3 photons/cm2/s ~ 2.4e-8 erg/cm2/s
maxi_conv = 2.4e-8 / 3.3
maxi_df["X_FLUX"] = maxi_df["RATE_2_20"] * maxi_conv
maxi_df["X_FLUX_ERR"] = maxi_df["ERR_2_20"] * maxi_conv

# ==========================================
# 2. Fetch and Clean ZTF Optical Data
# ==========================================
print("Fetching ZTF data from ALeRCE...")
alerce = Alerce()
ra, dec = 261.9305, -16.2052
objects = alerce.query_objects(ra=ra, dec=dec, radius=15, format="pandas")
oid = objects["oid"].iloc[0]
ztf_df = alerce.query_detections(oid, format="pandas")

mag_col = "magpsf" if "magpsf" in ztf_df.columns else "mag"
err_col = "sigmapsf" if "sigmapsf" in ztf_df.columns else "e_mag"

ztf_df = ztf_df.dropna(subset=["mjd", mag_col, err_col])
ztf_df = ztf_df[(ztf_df["mjd"] > 60150) & (ztf_df["mjd"] < 60400)]

# CONVERSION: AB Magnitude to Flux Density (mJy)
def mag_to_flux(mag, mag_err):
    flux = 3631 * 10**(-0.4 * mag) * 1000  # Jy to mJy
    flux_err = flux * (mag_err / 1.0857)
    return flux, flux_err

ztf_df["OPT_FLUX"], ztf_df["OPT_FLUX_ERR"] = mag_to_flux(ztf_df[mag_col], ztf_df[err_col])

g_band = ztf_df[ztf_df["fid"] == 1]
r_band = ztf_df[ztf_df["fid"] == 2]

# ==========================================
# 3. Model Fitting (FRED and Reprocessing)
# ==========================================
def fred_model(t, t_start, t_rise, t_decay, amplitude):
    flux = np.zeros_like(t)
    mask = t > t_start
    t_eff = t[mask] - t_start
    norm_factor = np.exp(2 * np.sqrt(t_rise / t_decay))
    flux[mask] = amplitude * norm_factor * np.exp(-(t_rise / t_eff) - (t_eff / t_decay))
    return flux

# Fit FRED to MAXI Data
initial_guesses = [60170, 5.0, 50.0, maxi_df["X_FLUX"].max()]
bounds = ([60160, 0.1, 10.0, 0], [60180, 20.0, 300.0, 1e-6])

popt_xray, _ = curve_fit(
    fred_model, maxi_df["MJD"], maxi_df["X_FLUX"], 
    p0=initial_guesses, bounds=bounds, sigma=maxi_df["X_FLUX_ERR"]
)

# Fit Reprocessing Scaling Factor to ZTF r-band
# F_opt = C * (F_x_model)^0.5
def reprocessing_model(t, scale_factor):
    xray_flux_at_t = fred_model(t, *popt_xray)
    return scale_factor * (xray_flux_at_t ** 0.5)

# Fit this specifically to the r-band data
popt_opt, _ = curve_fit(
    reprocessing_model, r_band["mjd"], r_band["OPT_FLUX"], 
    p0=[1e5], sigma=r_band["OPT_FLUX_ERR"]
)
best_scale_factor = popt_opt[0]

# Generate smooth arrays for plotting
model_time = np.linspace(60150, 60400, 1000)
xray_fit_curve = fred_model(model_time, *popt_xray)
simulated_opt_curve = reprocessing_model(model_time, best_scale_factor)

# ==========================================
# 4. Plotting
# ==========================================
fig, ax1 = plt.subplots(figsize=(14, 8))

# MAXI X-ray Data and Fit
color1 = 'dimgray'
ax1.errorbar(
    maxi_df["MJD"], maxi_df["X_FLUX"], yerr=maxi_df["X_FLUX_ERR"],
    fmt="o", color=color1, ms=4, alpha=0.6, label="MAXI Data (2-20 keV)"
)
ax1.plot(
    model_time, xray_fit_curve, color="black", linewidth=2.5, 
    label=f"FRED Fit ($t_{{rise}}$={popt_xray[1]:.1f}d, $t_{{decay}}$={popt_xray[2]:.1f}d)"
)

ax1.set_xlabel("Time (MJD)", fontsize=12)
ax1.set_ylabel("X-ray Flux (erg cm$^{-2}$ s$^{-1}$)", color="black", fontsize=12)
ax1.tick_params(axis="y", labelcolor="black")
ax1.grid(True, linestyle="--", alpha=0.5)

# ZTF Optical Data and Simulation
ax2 = ax1.twinx()

if not g_band.empty:
    ax2.errorbar(
        g_band["mjd"], g_band["OPT_FLUX"], yerr=g_band["OPT_FLUX_ERR"],
        fmt="o", color="forestgreen", ms=5, alpha=0.8, label="ZTF g-band Data"
    )
if not r_band.empty:
    ax2.errorbar(
        r_band["mjd"], r_band["OPT_FLUX"], yerr=r_band["OPT_FLUX_ERR"],
        fmt="o", color="firebrick", ms=5, alpha=0.8, label="ZTF r-band Data"
    )

ax2.plot(
    model_time, simulated_opt_curve, color="orange", linewidth=2.5, linestyle="--",
    label=f"Reprocessing Fit ($C \\times F_X^{{0.5}}$)"
)

ax2.set_ylabel("Optical Flux Density (mJy)", color="darkred", fontsize=12)
ax2.tick_params(axis="y", labelcolor="darkred")
ax1.set_xlim(60160, 60350)

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=11, framealpha=0.9)

plt.title("SWIFT J1727.8-1613: MAXI X-ray FRED Fit & Optical Reprocessing", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()
plt.show()